Import necessary libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import weather data

In [5]:
import wetterdienst
print(wetterdienst.__version__)

0.119.0


In [ ]:
from wetterdienst import Settings
from wetterdienst.provider.dwd.observation import DwdObservationRequest, DwdObservationParameter, DwdObservationResolution, DwdObservationPeriod
import pandas as pd
import numpy as np

# ── Configuration ──────────────────────────────────────────────────────────────

# Stations spread across Germany for national coverage:
# Hamburg (north/wind), Bremen, Berlin, Düsseldorf, Frankfurt,
# Munich, Stuttgart, Leipzig, Nuremberg, Cologne,
# Flensburg (north coast), Zugspitze excluded (too alpine),
# Dresden, Hanover, Karlsruhe
STATION_IDS = [
    "01975",  # Hamburg-Fuhlsbüttel
    "00691",  # Bremen
    "00433",  # Berlin-Tempelhof
    "02074",  # Düsseldorf (Düsseldorf-Flughafen)
    "01420",  # Frankfurt/Main
    "03668",  # Munich (München-Stadt)
    "04931",  # Stuttgart
    "02932",  # Leipzig
    "03730",  # Nuremberg (Nürnberg)
    "02564",  # Cologne/Bonn
    "01050",  # Flensburg
    "01048",  # Dresden
    "02564",  # Hanover → replace with "02564" ... 
]

PARAMETERS = [
    DwdObservationParameter.HOURLY.TEMPERATURE_AIR_MEAN_200,   # °C at 2m
    DwdObservationParameter.HOURLY.WIND_SPEED,                  # m/s
    DwdObservationParameter.HOURLY.RADIATION_GLOBAL,            # W/m²
    DwdObservationParameter.HOURLY.PRECIPITATION_HEIGHT,        # mm
    DwdObservationParameter.HOURLY.CLOUD_COVER_TOTAL,           # oktas (0–8)
    DwdObservationParameter.HOURLY.PRESSURE_AIR_SITE,           # hPa
]

TRAIN_START = "2018-01-01"
TRAIN_END   = "2022-12-31"
TEST_START  = "2023-01-01"
TEST_END    = "2024-12-31"

# ── Fetch raw observations ─────────────────────────────────────────────────────

settings = Settings(ts_shape="long", ts_humanize=True, ts_convert_units=True)

request = DwdObservationRequest(
    parameter=PARAMETERS,
    resolution=DwdObservationResolution.HOURLY,
    start_date=TRAIN_START,
    end_date=TEST_END,
    settings=settings,
)

# Filter to our station set
stations = request.filter_by_station_id(station_id=STATION_IDS)

print("Fetching observations (this may take a few minutes)...")
df_raw = stations.values.all().df
print(f"Raw shape: {df_raw.shape}")

# ── Clean and pivot ────────────────────────────────────────────────────────────

# Drop null values and rename for clarity
df_clean = df_raw.dropna(subset=["value"]).copy()
df_clean["date"] = pd.to_datetime(df_clean["date"]).dt.tz_localize(None)

# Pivot: one column per parameter per station
df_pivot = df_clean.pivot_table(
    index=["date", "station_id"],
    columns="parameter",
    values="value",
    aggfunc="mean",  # handles rare duplicate timestamps
).reset_index()

df_pivot.columns.name = None
df_pivot.columns = [str(c).lower().replace(" ", "_") for c in df_pivot.columns]

# ── Spatial aggregation: national hourly mean ──────────────────────────────────
# Simple mean across all stations — sufficient for a school project;
# production would weight by Voronoi area or population

PARAM_COLS = [
    "temperature_air_mean_200",
    "wind_speed",
    "radiation_global",
    "precipitation_height",
    "cloud_cover_total",
    "pressure_air_site",
]
# Keep only the parameter columns that actually exist after the pivot
existing_params = [c for c in PARAM_COLS if c in df_pivot.columns]

df_national = (
    df_pivot.groupby("date")[existing_params]
    .mean()
    .reset_index()
    .sort_values("date")
    .set_index("date")
)

# ── Resample to complete hourly index, interpolate gaps ───────────────────────
full_idx = pd.date_range(start=TRAIN_START, end=TEST_END + " 23:00", freq="h")
df_national = df_national.reindex(full_idx)

# Linear interpolation for short gaps (≤3h); longer gaps stay NaN
df_national = df_national.interpolate(method="linear", limit=3)

# Report coverage
coverage = df_national.notna().mean() * 100
print("\nFeature coverage (%):")
print(coverage.round(1).to_string())

# ── Derived features ───────────────────────────────────────────────────────────
df_national["wind_power_proxy"]  = df_national["wind_speed"] ** 3          # ∝ wind turbine output
df_national["solar_proxy"]       = df_national["radiation_global"].clip(lower=0)
df_national["heating_degree"]    = (18 - df_national["temperature_air_mean_200"]).clip(lower=0)
df_national["cooling_degree"]    = (df_national["temperature_air_mean_200"] - 24).clip(lower=0)

# ── Calendar features ──────────────────────────────────────────────────────────
df_national["hour"]        = df_national.index.hour
df_national["weekday"]     = df_national.index.dayofweek          # 0=Mon, 6=Sun
df_national["month"]       = df_national.index.month
df_national["is_weekend"]  = (df_national["weekday"] >= 5).astype(int)
df_national["hour_sin"]    = np.sin(2 * np.pi * df_national["hour"] / 24)
df_national["hour_cos"]    = np.cos(2 * np.pi * df_national["hour"] / 24)
df_national["month_sin"]   = np.sin(2 * np.pi * df_national["month"] / 12)
df_national["month_cos"]   = np.cos(2 * np.pi * df_national["month"] / 12)

# ── Train / test split ─────────────────────────────────────────────────────────
df_train = df_national[TRAIN_START:TRAIN_END]
df_test  = df_national[TEST_START:TEST_END]

print(f"\nTrain shape : {df_train.shape}  ({df_train.index[0].date()} → {df_train.index[-1].date()})")
print(f"Test  shape : {df_test.shape}  ({df_test.index[0].date()} → {df_test.index[-1].date()})")
print(f"\nColumns:\n{list(df_national.columns)}")

ImportError: cannot import name 'DwdObservationParameter' from 'wetterdienst.provider.dwd.observation' (/Users/sofiiavelykokhatska/Predictor-of-Energy-Market-Price-based-on-Weather/.venv/lib/python3.12/site-packages/wetterdienst/provider/dwd/observation/__init__.py)